# How Special Are Legendary Pokémon - What Makes a Pokemon Legendary, and Can We Predict It?

**Author:** Maxi Hase
**Course:** DCS 211, Fall 2026

**Dataset:** "The Complete Pokemon Dataset" by Rounak Banik, found on Kaggle (https://www.kaggle.com/datasets/rounakbanik/pokemon)

This notebook is part of my final project for DCS 211, and seeks to explore whether there is any correlation between certain stats of a Pokemon and them being legendary, as well as whether we can predict whether a given Pokemon will be legendary.

In [7]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay)


## A. Data Collection and Exploration

In [8]:
df = pd.read_csv('data/pokemon.csv') # saving the CSV file as a dataframe
df.shape # 801 rows, 41 columns
df.head()

,abilities,against_bug,against_dark,against_dragon,against_electric,against_fairy,against_fight,against_fire,against_flying,against_ghost,...,percentage_male,pokedex_number,sp_attack,sp_defense,speed,type1,type2,weight_kg,generation,is_legendary
0,"['Overgrow', 'Chlorophyll']",1.0,1.0,1.0,0.5,0.5,0.5,2.0,2.0,1.0,...,88.1,1,65,65,45,grass,poison,6.9,1,0
1,"['Overgrow', 'Chlorophyll']",1.0,1.0,1.0,0.5,0.5,0.5,2.0,2.0,1.0,...,88.1,2,80,80,60,grass,poison,13.0,1,0
2,"['Overgrow', 'Chlorophyll']",1.0,1.0,1.0,0.5,0.5,0.5,2.0,2.0,1.0,...,88.1,3,122,120,80,grass,poison,100.0,1,0
3,"['Blaze', 'Solar Power']",0.5,1.0,1.0,1.0,0.5,1.0,0.5,1.0,1.0,...,88.1,4,60,50,65,fire,NaN,8.5,1,0
4,"['Blaze', 'Solar Power']",0.5,1.0,1.0,1.0,0.5,1.0,0.5,1.0,1.0,...,88.1,5,80,65,80,fire,NaN,19.0,1,0


In [9]:
df.dtypes

abilities                str
against_bug          float64
against_dark         float64
against_dragon       float64
against_electric     float64
against_fairy        float64
against_fight        float64
against_fire         float64
against_flying       float64
against_ghost        float64
against_grass        float64
against_ground       float64
against_ice          float64
against_normal       float64
against_poison       float64
against_psychic      float64
against_rock         float64
against_steel        float64
against_water        float64
attack                 int64
base_egg_steps         int64
base_happiness         int64
base_total             int64
capture_rate             str
classfication            str
defense                int64
experience_growth      int64
height_m             float64
hp                     int64
japanese_name            str
name                     str
percentage_male      float64
pokedex_number         int64
sp_attack              int64
sp_defense    

In [10]:
df.describe()

,against_bug,against_dark,against_dragon,against_electric,against_fairy,against_fight,against_fire,against_flying,against_ghost,against_grass,...,height_m,hp,percentage_male,pokedex_number,sp_attack,sp_defense,speed,weight_kg,generation,is_legendary
count,801.000000,801.000000,801.000000,801.000000,801.000000,801.000000,801.000000,801.000000,801.000000,801.000000,...,781.000000,801.000000,703.000000,801.000000,801.000000,801.000000,801.000000,781.000000,801.000000,801.000000
mean,0.996255,1.057116,0.968789,1.073970,1.068976,1.065543,1.135456,1.192884,0.985019,1.034020,...,1.163892,68.958801,55.155761,401.000000,71.305868,70.911361,66.334582,61.378105,3.690387,0.087391
std,0.597248,0.438142,0.353058,0.654962,0.522167,0.717251,0.691853,0.604488,0.558256,0.788896,...,1.080326,26.576015,20.261623,231.373075,32.353826,27.942501,28.907662,109.354766,1.930420,0.282583
min,0.250000,0.250000,0.000000,0.000000,0.250000,0.000000,0.250000,0.250000,0.000000,0.250000,...,0.100000,1.000000,0.000000,1.000000,10.000000,20.000000,5.000000,0.100000,1.000000,0.000000
25%,0.500000,1.000000,1.000000,0.500000,1.000000,0.500000,0.500000,1.000000,1.000000,0.500000,...,0.600000,50.000000,50.000000,201.000000,45.000000,50.000000,45.000000,9.000000,2.000000,0.000000
50%,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,65.000000,50.000000,401.000000,65.000000,66.000000,65.000000,27.300000,4.000000,0.000000
75%,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,2.000000,1.000000,1.000000,1.000000,...,1.500000,80.000000,50.000000,601.000000,91.000000,90.000000,85.000000,64.800000,5.000000,0.000000
max,4.000000,4.000000,2.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,...,14.500000,255.000000,100.000000,801.000000,194.000000,230.000000,180.000000,999.900000,7.000000,1.000000


In [11]:
df['is_legendary'].value_counts() # 0: 731, 1: 70

is_legendary
0    731
1     70
Name: count, dtype: int64

In [12]:
# this first checks each entry in the dataframe for whether 
# it has a missing value (NaN), then sums up the number of missing values per column, and finally
# sorts the columns by how many missing values exist
df.isna().sum().sort_values(ascending=False) 

type2                384
percentage_male       98
weight_kg             20
height_m              20
name                   0
capture_rate           0
classfication          0
defense                0
experience_growth      0
hp                     0
japanese_name          0
abilities              0
base_happiness         0
pokedex_number         0
sp_attack              0
sp_defense             0
speed                  0
type1                  0
generation             0
base_total             0
base_egg_steps         0
against_bug            0
attack                 0
against_dark           0
against_dragon         0
against_electric       0
against_fairy          0
against_fight          0
against_fire           0
against_flying         0
against_ghost          0
against_grass          0
against_ground         0
against_ice            0
against_normal         0
against_poison         0
against_psychic        0
against_rock           0
against_steel          0
against_water          0


## Summary of Section A:

**Size and Structure:** The dataset contains 801 rows along with 41 features. Each row represents one Pokémon, and the features include abilities, stats (e.g. Attack, Defense, etc.), type, capture rate, among others.

**Data Types:** Most features are numerical values, as would be expected. There are also some expected string features, such as the name, or the type. However, there is one string feature that should be numerical: the capture rate. This suggests that there is at least one Pokémon with a missing capture rate, which can either be a mistake in the dataset, or a quirk that a few Pokémon have. This will be investigated in Section B.

**Distribution of Legendary Pokémon:** There are 70 legendary Pokémon, and 731 non-legendary Pokémon. This makes about 8.7% of Pokémon in this Dataset legendary. This means two things for our context: For one, there is limited data to train on and to see patterns in, and secondly, the data is heavily skewed towards non-legendary Pokémon. If a model simply guessed that every given Pokémon is not legendary, it would be right 91.3% of the time, so this is the baseline that this prediction model would have to beat.

**Missing values:** There are four features with missing entries for some Pokémon. The two most frequent are to be expected: most notably, 384 Pokémon are missing an entry for a second type. This is most likely because these Pokémon only have one type, so there is no second type to list. For example, while Venusaur is of Type Grass and Poison, Turtoise only has Type Water. Additionally, there are 98 Pokémon without a percentage of male Pokémon (to female), which is also not quite as surprising, because some Pokémon don't have a gender, such as Porygon. 
The other two features are more curious, however. Namely, there are 20 missing entries both for height and weight. These will need to be investigated in the next section.